# Phase 3g: forward-looking-statement models against the phase 3 rules

Plan: `plans/phase_3g.md` (with two amendments before computation and one after). Computation: `analysis/phase3g_fls.py` (models, cross-tables, set S, blind spot check) and `analysis/phase3g_read.py` (reading of S, report search, classes with verified quotes, updated basis); outputs in `data/phase3g/`. Descriptive and exploratory: no test, no register entry.

**Note of 2026-10-09.** The count and the status in this notebook are superseded by phase 3h (version 2), phase 3i and phase 3j. The current state is in `FINDINGS.md`.

In [ ]:
from pathlib import Path
import json
import pandas as pd

ROOT = Path.cwd().parents[1] if Path.cwd().name == "roman" else Path.cwd()
OUT = ROOT / "notebooks" / "roman" / "data" / "phase3g"
pd.set_option("display.max_colwidth", 200)

## Models

In [ ]:
m = json.loads((OUT / "models.json").read_text())
pd.DataFrame([("primary", *m["finbert"], ", ".join(m["finbert_labels"].values())),
              ("second opinion", *m["roberta"], ", ".join(m["roberta_labels"].values()))],
             columns=["role", "model", "revision", "labels"])

Both models ran locally on the GPU with a maximum length of 256 tokens. FLS-RoBERTa's placeholder label "label" is ignored; a sentence counts as FLS when the FLS probability exceeds the non_FLS probability.

## Cross-tables

In [ ]:
ct = pd.read_csv(OUT / "crosstabs.csv")
def table(row, col):
    t = ct[(ct.row == row) & (ct.col == col)]
    return t.pivot_table(index=["call", "row_value"], columns="col_value", values="sentences", fill_value=0) \
            .rename_axis(index=["call", row], columns=col)
table("candidate", "specific")

FinBERT-FLS labels 85 of the 105 UBS phase 3 candidates Specific FLS, and 495 further UBS sentences that the rules do not select. In the JPMorgan event call it labels none of the three candidates Specific but 17 other sentences.

In [ ]:
table("candidate", "roberta_fls")

In [ ]:
table("specific", "roberta_fls")

FLS-RoBERTa, which does not separate specific from non-specific statements, labels almost every FinBERT Specific FLS sentence as FLS (573 of 580 at UBS, 17 of 17 in the event call) and about a fifth of the rest as well. As a second opinion it therefore narrows nothing: it agrees with FinBERT on the specific statements and adds generic ones.

## Recall in the other direction (counted, not read)

The 20 UBS candidates and 3 event-call candidates that FinBERT does not label Specific FLS are counted only, as the plan fixes. The phase 3 candidates were read in phases 3 and 3d; these counts say only that the model would have missed about a fifth of them.

## The set S

In [ ]:
s = pd.read_csv(OUT / "set_s_read.csv", keep_default_na=False)
pd.crosstab([s.call, s.failed_conditions], s.roberta_fls, margins=True)

S has 73 sentences (68 UBS, 5 event call), all read because S is below the cap of 150. Most failed the horizon condition of the rules (the time reference lies in the context or is phrased as "for the foreseeable future", "over the cycle" or "in the coming months"); FLS-RoBERTa labels all 73 as FLS.

## Reading of S

In [ ]:
s["result"] = s.genuine.map({"no": "not genuine", "yes": "genuine"})
s.loc[(s.genuine == "yes") & (s.duplicate_of != "new"), "result"] = "genuine, duplicate"
s.loc[s.duplicate_of == "new", "result"] = "genuine, new"
pd.crosstab(s.call, s.result, margins=True)

Of the 73 sentences, 53 are genuine commitments and 20 are not; 31 of the genuine ones restate a known unit and 22 are new, all from UBS. The event call adds nothing: its two genuine sentences are E766 and T6 of phase 3d.

In [ ]:
pd.crosstab(s.failed_conditions, s.result)

In [ ]:
s[s.genuine == "no"][["stmt_id", "call", "quarter", "text", "comment"]]

Of the 20 sentences that are not genuine, 6 carry a figure for the present or the past while the forward-looking part has none, 4 are pro forma figures at the set-up of NCL or at the First Republic closing, 4 are actual values reported after the call, 2 are contractual transaction terms and 2 are central bank rate assumptions. The remaining two are a withdrawn illustration (4955) and a target without a horizon (6621, borderline).

In [ ]:
s[(s.genuine == "yes") & (s.duplicate_of != "new")][["stmt_id", "quarter", "duplicate_of", "text", "comment"]]

The duplicates show the main weakness of the rules: they restate known targets in wording without a date in the same sentence (CET1 around 14%, Basel III 5%). Sentence 3691 states the NCL reduction of T14 one quarter before its phase 3 origin.

## New commitments and their report counterparts

In [ ]:
r = pd.read_csv(OUT / "new_units_read.csv", keep_default_na=False)
r[["unit", "type", "quarter", "description", "class", "class_all", "class_pillar3", "file", "page"]]

In [ ]:
pd.crosstab(r.type, r["class"], margins=True)

Of the 22 new commitments, 12 have the same figure in a report of window W1 (A), two are forward-looking there with another or no figure (B) and 8 are not found (C). The eight targets split into five A, one B (F12, after the independent manual check) and two C; the six C cases among guidance are mostly short-term details (a funding benefit in Group Items, a hedge-related NII effect, integration expenses for one quarter).

In [ ]:
r[["unit", "quote", "file", "page", "comment"]]

## Updated comparison basis (UBS targets set after the acquisition)

In [ ]:
pd.read_csv(OUT / "basis.csv")

With T27 and T16 moved to B by the consistency rule, the phase 3d basis is 16 A, 13 B, 2 C; adding the eight new targets gives 21 A, 14 B, 4 C of 39 in W1. The share without any report counterpart is 6% before and 10% after adding the new targets.

## New candidate call-only targets

In [ ]:
r[r.unit.isin(["F11", "F12", "F22"])][["unit", "quarter", "call_sentence", "class", "class_all", "class_pillar3", "borderline_quote", "comment"]]

Three new targets were not found in any UBS quarterly or annual report or in the available Pillar 3 reports: a going concern capital ratio of around 18% built with AT1 (F11), a reduction of Group LRD by over 100 billion over three years (F12) and an LCR below the 4Q24 level of 188% (F22). The independent manual check confirmed F11 and F22 as C and set F12 to B, because the annual reports list the NCL priority "Reduce RWA and LRD" without a figure; this check read the borderline cases, not every hit.

In [ ]:
r[r.unit.isin(["F11", "F12", "F22"])][["unit", "class", "class_all", "class_pillar3", "checked_by_roman"]]

## Figures only in the call (UBS targets set after the acquisition, window ALL)

In [ ]:
d3 = pd.read_csv(ROOT / "notebooks" / "roman" / "data" / "phase3d" / "counterparts_read.csv", keep_default_na=False)
d3 = d3[(d3.bank == "UBS") & (d3.genuine == "yes") & (d3.type == "target") & (d3.pre_acquisition != "yes")]
t = pd.concat([d3[["unit", "class_all", "figure_call", "figure_report"]].assign(phase="3d"),
               r[r.type == "target"][["unit", "class_all", "description"]].rename(columns={"description": "figure_call"})
               .assign(figure_report=lambda x: x.unit.map({"F12": "NCL: reduce RWA and LRD, no figure"}).fillna(""), phase="3g")])
no_fig = {"T16", "T25", "T27", "T32", "T30", "F12"}  # report quote without any figure (read from figure_report and the quotes)
t["in the reports"] = t.class_all.map({"A": "same figure"})
t.loc[t.class_all == "C", "in the reports"] = "not mentioned"
t.loc[(t.class_all == "B") & t.unit.isin(no_fig), "in the reports"] = "mentioned without a figure"
t.loc[(t.class_all == "B") & ~t.unit.isin(no_fig), "in the reports"] = "mentioned with another figure"
t["figure only in the call"] = t["in the reports"].isin(["not mentioned", "mentioned without a figure"])
pd.crosstab(t["in the reports"], t["figure only in the call"], margins=True)

In [ ]:
t[t.class_all != "A"].sort_values(["in the reports", "unit"])[["unit", "phase", "in the reports", "figure_call", "figure_report"]]

Under the counting rule of phase 3d, 10 of the 39 targets have a figure only in the call: 4 are not mentioned in any report (T33, T68, F11, F22) and 6 are mentioned without a figure (T16, T25, T27, T32, F12 and the integration costs T30, reported separately). Six further targets are mentioned with another figure and are shown separately, not counted as call-only; T64 is a borderline assignment among them (structural 23% instead of the call's 20% for 2025).

In [ ]:
cu = json.loads((OUT / "check_units.json").read_text())
cu2 = json.loads((OUT / "check_units_2024q2.json").read_text())
pd.DataFrame([(u, len(v["files"]), len(v["terms"]), u in cu2) for u, v in cu.items()],
             columns=["unit", "files", "terms", "in 2024-Q2 Pillar 3 file"])

The check files follow the phase 3f format; unlike in phase 3f they were written after the search, because the units were only known then. The output of the independent check is in `data/phase3g/roman_check/`. The 2024-Q2 Pillar 3 report is not yet available and is searched for F11 and F12 once it is added.

## One divergence (exploratory)

In [ ]:
r[r.unit == "F19"][["call_sentence", "quote", "file", "comment"]]

For the sale of the US mortgage servicing business the call gives a gain of around 100 million in 2025-Q1, while every report from 2024-Q2 to the 2024 annual report states that no material profit or loss is expected. Both can be true at UBS's scale, but only the call quantifies the effect.

## Blind spot check

Ten sentences of S (seed 1) were written to `data/labelling/phase3g_spotcheck.csv` without model scores or the first reading and labelled blind by a second reader; the comparison is in `data/phase3g/spotcheck_comparison.csv`.

In [ ]:
sc = pd.read_csv(OUT / "spotcheck_comparison.csv", keep_default_na=False)
agr = json.loads((OUT / "spotcheck_agreement.json").read_text())
display(pd.Series(agr).to_frame("value"))
sc[["item_id", "stmt_id", "genuine_roman", "genuine_mine", "duplicate_of_roman", "duplicate_of_mine"]]

The two readings agree on 9 of 10 sentences; Cohen's kappa is 0 because the second reader labelled all ten genuine, so agreement equals chance agreement and kappa carries no information here. The one disagreement is 6150 (some but not all of the 30 basis points of Basel III impact in the parent bank): the second reader counts it as genuine, the first reading did not, but it would be a duplicate of the Group effect (candidate 5907) and changes no new unit or class. The second reader left `duplicate_of` empty, so the duplicate assignment is not checked.

## Limits

FinBERT-FLS was trained on US 10-K MD&A sentences, not on calls, and neither model is validated on these data. S was read by one reader; C means not found with the pre-specified terms, and the first reading pass missed three counterparts in truncated hit text, corrected in a second full-text pass (plan amendment). Sentences are classified without context, so a commitment whose figure sits in another sentence is missed by both methods.